In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, balanced_accuracy_score

# ---------------------------------------------------------
# 1. CARGAR DATOS (KDD no tiene encabezados)
# ---------------------------------------------------------
print("Cargando datasets de KDD...")
train = pd.read_csv('train_100000.csv', header=None)
val = pd.read_csv('validation.csv', header=None)

# ---------------------------------------------------------
# 2. SEPARAR X y y
# ---------------------------------------------------------
# En KDD, las primeras 41 columnas son características y la ÚLTIMA es el target.
X_train = train.iloc[:, :-1]
y_train_raw = train.iloc[:, -1]

X_val = val.iloc[:, :-1]
y_val_raw = val.iloc[:, -1]

# Convertir el target: 'normal.' = 0, cualquier otra cosa (ataque) = 1
y_train = (y_train_raw != 'normal.').astype(int)
y_val = (y_val_raw != 'normal.').astype(int)

# ---------------------------------------------------------
# 3. PREPROCESSING (Reglas de Edgar para KDD)
# ---------------------------------------------------------
print("Aplicando OneHotEncoding y StandardScaler...")
# Índices de columnas categóricas (1, 2, 3) y numéricas (el resto)
categorical_features = [1, 2, 3]
numerical_features = [0] + list(range(4, X_train.shape[1]))

# ColumnTransformer permite aplicar distintos procesos a distintas columnas a la vez
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# FIT + TRANSFORM en train, SOLO TRANSFORM en validation
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

# ---------------------------------------------------------
# 4. ENTRENAR MODELO (Ej. Random Forest)
# ---------------------------------------------------------
print("Entrenando modelo...")
start_time = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    class_weight='balanced', # KDD tiene más ataques que normal, esto ayuda
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train_processed, y_train)
training_time = time.time() - start_time

# ---------------------------------------------------------
# 5. EVALUAR EN VALIDATION
# ---------------------------------------------------------
print("Evaluando modelo...")
y_pred = rf_model.predict(X_val_processed)
# ROC-AUC necesita las probabilidades, no solo la clase final
y_pred_proba = rf_model.predict_proba(X_val_processed)[:, 1]

# ---------------------------------------------------------
# 6. MÉTRICAS PARA EL SHEET DE KDD
# ---------------------------------------------------------
acc = accuracy_score(y_val, y_pred)
rec = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
roc_auc = roc_auc_score(y_val, y_pred_proba)
bal_acc = balanced_accuracy_score(y_val, y_pred)

print("-" * 40)
print("RESULTADOS PARA EL SHEET DE KDD:")
print(f"Accuracy:          {acc:.4f}")
print(f"Recall de attack:  {rec:.4f}")
print(f"F1 de attack:      {f1:.4f}")
print(f"ROC-AUC:           {roc_auc:.4f}")
print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"Tiempo de entreno: {training_time:.2f} segundos")
print("-" * 40)

# ---------------------------------------------------------
# 7. GUARDAR PREDICCIONES
# ---------------------------------------------------------
# Usaremos un índice estándar que comience en 1 para el observation_id
val_ids = np.arange(1, len(val) + 1)

predictions_df = pd.DataFrame({
    'observation_id': val_ids,
    'prediction': y_pred
})
predictions_df.to_csv('kdd_validation_predictions.csv', index=False)
print("Predicciones guardadas en 'kdd_validation_predictions.csv'")

Cargando datasets de KDD...
Aplicando OneHotEncoding y StandardScaler...
Entrenando modelo...
Evaluando modelo...
----------------------------------------
RESULTADOS PARA EL SHEET DE KDD:
Accuracy:          0.9988
Recall de attack:  0.9992
F1 de attack:      0.9987
ROC-AUC:           1.0000
Balanced Accuracy: 0.9988
Tiempo de entreno: 7.82 segundos
----------------------------------------
Predicciones guardadas en 'kdd_validation_predictions.csv'


In [2]:
import pandas as pd
import numpy as np
import time
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, balanced_accuracy_score

# ---------------------------------------------------------
# 1. CARGAR DATOS
# ---------------------------------------------------------
print("Cargando datasets de KDD...")
train = pd.read_csv('train_100000.csv', header=None)
val = pd.read_csv('validation.csv', header=None)

# ---------------------------------------------------------
# 2. SEPARAR X y y
# ---------------------------------------------------------
X_train = train.iloc[:, :-1]
y_train_raw = train.iloc[:, -1]

X_val = val.iloc[:, :-1]
y_val_raw = val.iloc[:, -1]

# Convertir el target: 'normal.' = 0, ataque = 1
y_train = (y_train_raw != 'normal.').astype(int)
y_val = (y_val_raw != 'normal.').astype(int)

# ---------------------------------------------------------
# 3. PREPROCESSING
# ---------------------------------------------------------
print("Aplicando OneHotEncoding y StandardScaler...")
categorical_features = [1, 2, 3]
numerical_features = [0] + list(range(4, X_train.shape[1]))

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# FIT + TRANSFORM en train, SOLO TRANSFORM en validation
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

# ---------------------------------------------------------
# 4. ENTRENAR MULTI-LAYER PERCEPTRON (MLP)
# ---------------------------------------------------------
print("Entrenando Red Neuronal (MLP)...")
start_time = time.time()

mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32), # Dos capas ocultas: 64 neuronas en la primera, 32 en la segunda
    activation='relu',           # Función de activación estándar para redes profundas
    solver='adam',               # Optimizador muy eficiente
    max_iter=300,                # Límite máximo de épocas
    early_stopping=True,         # Detiene el entrenamiento si no mejora para evitar overfitting
    random_state=42
)

mlp_model.fit(X_train_processed, y_train)
training_time = time.time() - start_time

# ---------------------------------------------------------
# 5. EVALUAR EN VALIDATION
# ---------------------------------------------------------
print("Evaluando modelo...")
y_pred = mlp_model.predict(X_val_processed)
# Probabilidades necesarias para el cálculo del ROC-AUC
y_pred_proba = mlp_model.predict_proba(X_val_processed)[:, 1]

# ---------------------------------------------------------
# 6. MÉTRICAS PARA EL SHEET DE KDD
# ---------------------------------------------------------
acc = accuracy_score(y_val, y_pred)
rec = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
roc_auc = roc_auc_score(y_val, y_pred_proba)
bal_acc = balanced_accuracy_score(y_val, y_pred)

print("-" * 40)
print("RESULTADOS PARA EL SHEET DE KDD (MLP):")
print(f"Accuracy:          {acc:.4f}")
print(f"Recall de attack:  {rec:.4f}")
print(f"F1 de attack:      {f1:.4f}")
print(f"ROC-AUC:           {roc_auc:.4f}")
print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"Tiempo de entreno: {training_time:.2f} segundos")
print("-" * 40)

# ---------------------------------------------------------
# 7. GUARDAR PREDICCIONES
# ---------------------------------------------------------
val_ids = np.arange(1, len(val) + 1)

predictions_df = pd.DataFrame({
    'observation_id': val_ids,
    'prediction': y_pred
})
predictions_df.to_csv('mlp_kdd_validation_predictions.csv', index=False)
print("Predicciones guardadas en 'mlp_kdd_validation_predictions.csv'")

Cargando datasets de KDD...
Aplicando OneHotEncoding y StandardScaler...
Entrenando Red Neuronal (MLP)...
Evaluando modelo...
----------------------------------------
RESULTADOS PARA EL SHEET DE KDD (MLP):
Accuracy:          0.9979
Recall de attack:  0.9999
F1 de attack:      0.9987
ROC-AUC:           0.9999
Balanced Accuracy: 0.9949
Tiempo de entreno: 11.59 segundos
----------------------------------------
Predicciones guardadas en 'mlp_kdd_validation_predictions.csv'
